# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas
**Material:** *Preparar y estructurar datos con Python*, lecciones 2, 3 y 4
**Herramientas:** Python · Pandas · PyArrow · Jupyter Notebook

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno continúa el proyecto de la actividad anterior ([`data_quality.ipynb`](data_quality.ipynb)), donde se exploraron los cinco archivos sin modificarlos. Aquí se resuelven los problemas encontrados y se preparan los datos para el análisis.

**Contenido**

0. Preparación del entorno y carga de los datos
1. Lección 2 · Limpieza y estandarización
   - Ejemplos de la lección
   - Laboratorio guiado 2: pipeline de limpieza con informe de cambios
   - Reto 2
   - Documentación de la limpieza
2. Lección 3 · Filtrado, ordenamiento y agrupación
   - Ejemplos de la lección
   - Laboratorio guiado 3: perfil de hábitos de compra
   - Reto 3
   - Interpretación de los resultados
3. Lección 4 · Combinación de datasets
   - Ejemplos de la lección
   - Laboratorio guiado 4: tabla de análisis por producto
   - Reto 4
   - Validación y archivos procesados
4. Conclusiones

> **Cómo ejecutar:** copie los cinco CSV en `data/raw/`, abra este cuaderno desde `notebooks/` y use *Kernel → Restart & Run All*. Las lecciones se ejecutan en orden porque cada una usa las variables de la anterior. Los archivos generados se guardan en `data/processed/`.

---
# 0. Preparación del entorno y carga de los datos

Las rutas son relativas a `notebooks/`: los datos originales se leen de `data/raw/` y los resultados se guardan en `data/processed/`. Los archivos usan punto y coma (`;`) como separador.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

print('pandas', pd.__version__)

pandas 3.1.0


In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Las cinco tablas tienen los tamaños originales, los mismos de la actividad anterior. El diagnóstico de calidad está en [`reports/data_quality.md`](../reports/data_quality.md): 15 pedidos duplicados, nulos en tres columnas, columnas enteras leídas como `float64` y el departamento centinela `missing`. Esa es la lista de tareas de la Lección 2.

---
# 1. Lección 2 · Limpieza y estandarización

Para cada problema hay tres opciones: **eliminar, corregir o conservar**. La decisión depende de qué significa el dato. El orden de trabajo es: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

## 1.1 Ejemplos de la lección

### Duplicados exactos: `duplicated` y `drop_duplicates`

Primero se detectan los duplicados, sin borrar nada, para ver qué son.

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


**Resultado:** 30 filas, es decir, 15 pedidos y sus 15 copias. Todas tienen `order_dow = 3` y `order_hour_of_day = 2` (miércoles a las 2 a. m.). Que todas ocurran en la misma franja apunta a un fallo del sistema y no a pedidos reales, así que es seguro **eliminarlas**.

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

Las columnas enteras se convierten al tipo más pequeño que admite sus valores. Las que tienen nulos usan `Int16` (con mayúscula), el tipo entero *nullable*.

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


**Resultado:** la memoria de `orders` baja de 23,0 MB a 7,2 MB sin perder información. `int8` alcanza para días (0–6) y horas (0–23), `int16` para `order_number` (máximo 100) e `int32` para los identificadores.

### Nulos: comprobar la causa antes de decidir

Hipótesis: `days_since_prior_order` es nulo solo en el primer pedido de cada cliente.

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Resultado:** la hipótesis se cumple (`True`): cada nulo de `days_since_prior_order` es un primer pedido, así que se **conserva**; rellenarlo con 0 inventaría que el cliente volvió el mismo día y bajaría la espera media. Los 1.258 productos sin nombre están todos en el pasillo 100 y el departamento 21 (`missing`); se **corrigen** con `'Unknown'` porque el producto existe y no se puede borrar.

| Decisión | Columna | Causa |
|---|---|---|
| Conservar el nulo | `days_since_prior_order` | Primer pedido del cliente: no existe pedido anterior |
| Rellenar con `'Unknown'` | `product_name` | Producto sin datos de catálogo (pasillo y departamento `missing`) |
| Conservar el nulo | `add_to_cart_order` | Se investiga en el Reto 2, punto 3 |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Resultado:** había 403 nombres con espacios dobles y 16 con el espacio invisible `\xa0`; después de limpiar quedan 0 y 0. El reemplazo de `\xa0` va en un paso propio porque en pandas 3 la expresión `\s` no lo reconoce.

### Duplicados implícitos

Algunos nombres solo se repiten después de estandarizar, como `American Cheese slices` y `American Cheese Slices`.

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


**Resultado:** 141 nombres repetidos, y 268 de los 280 productos implicados aparecen en pedidos. La decisión es **conservar y documentar**: tienen `product_id` distintos y borrarlos dejaría pedidos apuntando a productos inexistentes. En los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


Aplicada a una tabla que ya está limpia, la función no cambia nada: la limpieza es repetible. Verificación de toda la limpieza (si la celda no muestra nada, todo está correcto):

In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## 1.2 Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir un *pipeline*, una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso. Se trabaja sobre copias nuevas de los archivos, sin tocar las tablas ya limpias.

**Paso 1. Recargar los datos crudos**

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6)

 (49694, 4) (4545007, 4)


**Paso 2. Una función que anota el estado de la tabla**

`anotar` guarda una "foto" de la tabla (filas, nulos y memoria) y la devuelve sin cambios, para poder intercalarla entre pasos.

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3. Un paso de limpieza por función**

Cada función recibe un DataFrame y devuelve otro; esa regla es la que permite encadenarlas.

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4. Encadenar los pasos con `pipe`**

In [15]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5. El informe de cambios**

In [16]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- La limpieza de nombres eliminó los 1.258 nulos de `products` sin perder filas.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos.

**Paso 6. Validar el resultado**

In [17]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


**Resultado:** `Validación superada` y `True True`. El pipeline produce exactamente las mismas tablas que la limpieza manual de los ejemplos, así que la limpieza se puede repetir con una sola instrucción cuando lleguen datos nuevos.

**Conclusiones clave del laboratorio 2**

- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una función de registro produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "¿qué le hiciste a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## 1.3 Reto 2

### Punto 1. Duplicados y valores de `order_prods`

¿Tiene `order_prods` filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [18]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1? True


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (0) ni pares `order_id` + `product_id` repetidos (0): cada producto aparece una sola vez por pedido. `reordered` solo contiene 0 y 1, así que no necesita corrección.

### Punto 2. Nombres de `aisles` y `departments`

¿Hay nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [19]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nombres duplicados:', texto.duplicated().sum())
    print('  duplicados sin distinguir mayúsculas:', texto.str.lower().str.strip().duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())

aisles
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
departments
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0


**Respuesta:** ni `aisles` (134 pasillos) ni `departments` (21 departamentos) tienen nombres duplicados, tampoco al ignorar mayúsculas, ni espacios al inicio, al final o dobles. **No necesitan limpieza de texto.** Lo único a tener en cuenta es el valor centinela `missing`, que es un dato válido como texto.

### Punto 3. Los 836 nulos de `add_to_cart_order`

¿En cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno de esos pedidos? ¿Qué los explica?

In [20]:
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
tamano_pedido = order_prods.groupby('order_id').size()

print('Pedidos con nulos:', len(pedidos_con_nulos))
print('Productos del pedido más pequeño con nulos:', tamano_pedido.loc[pedidos_con_nulos].min())
print('Máximo valor registrado de add_to_cart_order:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos:', (tamano_pedido > 64).sum())

Pedidos con nulos: 70
Productos del pedido más pequeño con nulos: 65
Máximo valor registrado de add_to_cart_order: 64
Pedidos con más de 64 productos: 70


**Respuesta:** los 836 nulos están en **70 pedidos**, y el más pequeño de ellos tiene **65 productos**. El valor máximo registrado de `add_to_cart_order` es 64 y hay exactamente 70 pedidos con más de 64 productos. Esto indica que el sistema solo registra la posición en el carrito hasta el puesto 64: a partir del artículo 65 la posición queda vacía. No es un error del pedido, sino un límite del registro, por eso se conservan los nulos.

### Punto 4. Nombres que cambiaron con la estandarización

Se carga de nuevo `products.csv` y se comparan los nombres no nulos antes y después de limpiar.

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

con_nombre = products_crudo['product_name'].notna()
cambiaron = products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']
print('Nombres que cambiaron:', cambiaron.sum(), 'de', con_nombre.sum())

ejemplos = pd.DataFrame({'antes': products_crudo.loc[con_nombre, 'product_name'][cambiaron],
                         'después': products.loc[con_nombre, 'product_name'][cambiaron]})
ejemplos.head(5)

Nombres que cambiaron: 6321 de 48436


,antes,después
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Respuesta:** **6.321 de los 48.436 nombres** con valor (13,1 %) cambiaron con la estandarización. La mayoría de los cambios son de mayúsculas (`with` → `With`, `mcg` → `Mcg`), además de los espacios dobles e invisibles.

### Punto 5. Día de la semana como categoría ordenada

In [22]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True),
                          name='dia')

print(dia_categoria.dtype)
print(dia_categoria.cat.categories.tolist())
print('Memoria como texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Memoria como categoría:', round(dia_categoria.memory_usage(deep=True) / 1e6, 2), 'MB')
print('¿domingo < sábado?', dia_categoria.min(), '<', dia_categoria.max())

category
['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']
Memoria como texto: 7.14 MB
Memoria como categoría: 0.48 MB
¿domingo < sábado? domingo < sábado


**Respuesta:** como texto, la columna del día ocupa 7,14 MB; como categoría ordenada, 0,48 MB, unas 15 veces menos, porque guarda cada nombre una sola vez y un código pequeño por fila. Además, al ser **ordenada**, pandas sabe que domingo va antes que sábado, así que `sort_values`, `min` y `max` respetan el orden de la semana y no el alfabético.

### Punto 6. Productos `'Unknown'` en los pedidos

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]

print('Filas de order_prods con productos Unknown:', len(filas_unknown))
print('% del total de filas:', round(len(filas_unknown) / len(order_prods) * 100, 2))
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())
print('% de los pedidos con productos:', round(filas_unknown['order_id'].nunique() / order_prods['order_id'].nunique() * 100, 2))

Filas de order_prods con productos Unknown: 9547
% del total de filas: 0.21
Pedidos en los que aparecen: 8254


% de los pedidos con productos: 1.83


**Respuesta:** hay **9.547 filas** de `order_prods` con productos `'Unknown'` (0,21 % del total), repartidas en **8.254 pedidos** (1,83 % de los pedidos con productos). **No los eliminaría:** son compras reales; borrarlas reduciría el tamaño de esos pedidos y alteraría la tasa de recompra y las posiciones en el carrito. Basta con identificarlos como `'Unknown'` y excluirlos solo en los análisis que dependan del nombre del producto.

### Punto 7. Reemplazar `'missing'` por un nulo real en una copia de `departments`

In [24]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print(departments_na.tail(3))
print('Nulos:', departments_na['department'].isna().sum())

# Efecto al agrupar: los productos del departamento nulo desaparecen del resultado
prod_depto = products.merge(departments_na, on='department_id')
print('Productos en total:', len(prod_depto))
print('Productos que quedan al agrupar:', prod_depto.groupby('department').size().sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Nulos: 1
Productos en total: 49694
Productos que quedan al agrupar: 48436


**Respuesta:** no conviene hacerlo en la tabla original porque `groupby` descarta los grupos nulos por defecto: al agrupar por departamento, los 1.258 productos de `missing` desaparecerían del resultado (quedan 48.436 de 49.694) y los totales dejarían de cuadrar.

## 1.4 Documentación de la limpieza

Resumen de las decisiones tomadas y su efecto. El informe del pipeline se guarda en `data/processed/cleaning_report.csv` y las tablas limpias en formato Parquet, que conserva los tipos de dato corregidos.

| Tabla | Problema | Decisión | Resultado |
|---|---|---|---|
| `orders` | 15 filas duplicadas (miércoles a las 2 a. m.) | Eliminar | 478.967 → 478.952 filas; `order_id` único |
| `orders` | Enteros leídos como `int64`/`float64` | Corregir tipos (`int8`/`int16`/`int32`, `Int16`) | 23,0 → 7,2 MB |
| `orders` | 28.817 nulos en `days_since_prior_order` | Conservar | Son primeros pedidos; rellenar con 0 falsearía la espera media |
| `products` | 1.258 nulos en `product_name` | Rellenar con `'Unknown'` | 0 nulos; los productos se conservan |
| `products` | Espacios dobles, espacios invisibles y mayúsculas irregulares | Estandarizar con `.str` | 6.321 nombres cambiados; 0 espacios dobles o invisibles |
| `products` | 141 nombres repetidos tras estandarizar | Conservar y documentar | Tienen `product_id` distintos; se agrupa por `product_id` |
| `order_prods` | Enteros en `int64`/`float64` | Corregir tipos | 145,4 → 54,5 MB |
| `order_prods` | 836 nulos en `add_to_cart_order` | Conservar | Posiciones no registradas en pedidos de más de 64 productos |
| `departments` | Valor centinela `missing` | Conservar como texto | Evita perder esos productos al agrupar |

In [25]:
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
orders.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products.to_parquet(SALIDA / 'products_clean.parquet', index=False)
order_prods.to_parquet(SALIDA / 'order_products_clean.parquet', index=False)

for archivo in sorted(SALIDA.glob('*')):
    if archivo.name != '.gitkeep':
        print(f'{archivo.name:32} {archivo.stat().st_size / 1e6:6.1f} MB')

cleaning_report.csv                 0.0 MB
customer_segments.csv               0.0 MB
order_products_clean.parquet       32.5 MB
orders_clean.parquet                5.4 MB
products_clean.parquet              1.5 MB
segment_time_profile.csv            0.0 MB


---
# 2. Lección 3 · Filtrado, ordenamiento y agrupación

Con las tablas limpias se responden preguntas: cuándo compran los clientes y con qué frecuencia vuelven. **Filtrar** elige las filas que importan, **ordenar** las jerarquiza y **agrupar** las resume. Antes de escribir código, cada pregunta se traduce a la frase "métrica por grupo".

## 2.1 Ejemplos de la lección

### Filtrado con máscaras booleanas

In [26]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [27]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


**Resultado:** hay 28.817 primeros pedidos (los mismos que los nulos de `days_since_prior_order`) y 45.599 pedidos hechos en fin de semana entre las 6 y las 11 de la mañana. Las condiciones se combinan con `&`, `|` y `~`, no con `and`/`or`.

### Filtrado con `query`

In [28]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


**Resultado:** 1.449 pedidos hechos desde las 10 p. m. por clientes que tardaron 30 días en volver. Dentro de `query` sí se escriben `and`, `or` y `not`.

### Ordenamiento con `sort_values` y `nlargest`

In [29]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


**Resultado:** tres clientes llegan al pedido número 100. En `historial`, el cliente 2 aparece primero con sus pedidos 1 y 4: el archivo no trae todos los pedidos de cada cliente.

### Agrupación con `groupby`

In [30]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


In [31]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


**Resultado:** la hora pico es las 10 a. m. con 40.578 pedidos. El domingo (0) y el lunes (1) son los días con más pedidos y clientes; la espera media entre pedidos va de 10,5 a 11,8 días según el día.

### Tabla dinámica con `pivot_table`

In [32]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
hora_max, dia_max = matriz.stack().idxmax()
print(f'Celda máxima: hora {hora_max}, día {dia_max} -> {matriz.max().max()} pedidos')
matriz.loc[[8, 10, 14, 20]]

(24, 7)
Celda máxima: hora 10, día 1 -> 7881 pedidos


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


**Resultado:** la matriz tiene forma `(24, 7)`, una fila por hora y una columna por día. La celda más alta es el lunes a las 10 h, con 7.881 pedidos.

### Valores por fila con `transform`

In [33]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


**Resultado:** 30.527 clientes tienen 5 pedidos o más y suman 231.946 pedidos. `transform` devuelve el conteo en cada fila original, a diferencia de `agg`, que devuelve una fila por grupo.

## 2.2 Laboratorio guiado 3: perfil de hábitos de compra

**Encargo:** el área de mercadeo quiere diseñar campañas según qué tan seguido compra cada cliente y necesita saber **a qué hora compra cada tipo de cliente**. El plan: contar los pedidos de cada cliente, clasificar a los clientes en tres segmentos, clasificar los pedidos en franjas horarias y cruzar segmentos y franjas.

**Paso 1. Pedidos por cliente**

In [34]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

**Resultado:** 157.437 clientes, con una mediana de 2 pedidos y un máximo de 28 en este archivo.

**Paso 2. Segmentar clientes con `pd.cut`**

Segmentos: **ocasional** (1 pedido), **regular** (2 a 4) y **frecuente** (5 o más).

In [35]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


**Paso 3. Llevar el segmento a cada pedido con `map`**

In [36]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


**Paso 4. Resumen por segmento**

In [37]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231.946 de 478.952) y vuelve cada 6 días (mediana). Los ocasionales tienen espera porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

**Paso 5. Franjas horarias y tabla cruzada**

In [38]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales). Para mercadeo, una campaña de fidelización debería llegar por la mañana.

**Paso 6. Las combinaciones de día y franja más activas**

In [39]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

**Resultado:** domingo en la tarde (42.340 pedidos), lunes en la tarde (37.432) y martes en la tarde (30.751) son las combinaciones más activas.

**Paso 7. Guardar el perfil y dejar `orders` como estaba**

Los resúmenes se guardan en `data/processed/` y las columnas auxiliares se eliminan para no arrastrarlas a la Lección 4.

In [40]:
resumen_segmento.to_csv(SALIDA / 'customer_segments.csv')
perfil.to_csv(SALIDA / 'segment_time_profile.csv')

orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


**Conclusiones clave del laboratorio 3**

- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

## 2.3 Reto 3

### Punto 1. Las tres horas con más pedidos los domingos

*Métrica por grupo:* número de pedidos por hora, solo en los pedidos con `order_dow == 0`.

In [41]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day').size().nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
dtype: int64

**Respuesta:** los domingos, las horas con más pedidos son las **15 h (7.585)**, las **14 h (7.533)** y las **13 h (7.483)**. A diferencia del total de la semana, donde el pico es a las 10 h, el domingo la demanda se desplaza a la primera parte de la tarde.

### Punto 2. Hora con mayor y menor espera media entre pedidos

*Métrica por grupo:* media de `days_since_prior_order` por hora.

In [42]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean().round(2)
print('Mayor espera:', espera_por_hora.idxmax(), 'h ->', espera_por_hora.max(), 'días')
print('Menor espera:', espera_por_hora.idxmin(), 'h ->', espera_por_hora.min(), 'días')

Mayor espera: 0 h -> 12.52 días
Menor espera: 22 h -> 10.62 días


**Respuesta:** la espera media es **mayor a las 0 h (12,52 días)** y **menor a las 22 h (10,62 días)**. Los pedidos de medianoche los hacen, en promedio, clientes que llevaban más tiempo sin comprar.

### Punto 3. Clientes con 10 pedidos o más

*Condición:* `pedidos_por_cliente >= 10`.

In [43]:
clientes_10 = (pedidos_por_cliente >= 10).sum()
print(clientes_10, 'clientes,', round(clientes_10 / len(pedidos_por_cliente) * 100, 2), '% del total')

6327 clientes, 4.02 % del total


**Respuesta:** **6.327 clientes** tienen 10 pedidos o más en este archivo, el 4,02 % de los 157.437 clientes.

### Punto 4. Pedidos hechos exactamente 7 días después del anterior

*Condición:* `days_since_prior_order == 7`.

In [44]:
a_los_7 = (orders['days_since_prior_order'] == 7).sum()
print(a_los_7, 'pedidos ->', round(a_los_7 / len(orders) * 100, 2), '% del total')
print('Días de espera más frecuentes:')
print(orders['days_since_prior_order'].value_counts().head(3))

44577 pedidos -> 9.31 % del total
Días de espera más frecuentes:
days_since_prior_order
30    51337
7     44577
6     33930
Name: count, dtype: Int64


**Respuesta:** **44.577 pedidos** se hicieron exactamente 7 días después del anterior, el **9,31 %** de los 478.952 pedidos. Es la segunda espera más frecuente después de 30 días (51.337), que probablemente agrupa todas las esperas de 30 días o más. El pico en 7 días muestra un hábito de compra semanal.

### Punto 5. Pedidos con 20 productos o más y el pedido más grande

*Métrica por grupo:* número de productos por pedido (`order_prods`).

In [45]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Respuesta:** **49.420 pedidos** tienen 20 productos o más. El pedido más grande es el **61355, con 127 productos**.

### Punto 6. Tasa de recompra en las posiciones 1 a 5 del carrito

*Métrica por grupo:* media de `reordered` por `add_to_cart_order`, con la condición `add_to_cart_order` entre 1 y 5.

In [46]:
primeras = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta:** la tasa de recompra baja a medida que avanza el carrito: 0,679 en la posición 1, 0,677 en la 2, 0,659 en la 3, 0,638 en la 4 y 0,617 en la 5. **Patrón:** los clientes agregan primero los productos que ya conocen y compran siempre; los productos nuevos tienden a ir más atrás en el carrito.

### Punto 7. Mediana de espera: fin de semana frente al resto de la semana

*Métrica por grupo:* mediana de `days_since_prior_order` por la condición "es fin de semana" (`order_dow` 0 o 6).

In [47]:
fin_de_semana = orders['order_dow'].isin([0, 6]).rename('fin_de_semana')
orders.groupby(fin_de_semana)['days_since_prior_order'].agg(['median', 'mean', 'count']).round(2)

,median,mean,count
fin_de_semana,,,
False,7.0,10.87,312767
True,8.0,11.62,137368


**Respuesta:** la mediana de espera es de **8 días en fin de semana** y de **7 días el resto de la semana** (medias de 11,62 y 10,87). Quienes compran el sábado o el domingo tardan algo más en volver; los que compran entre semana son más regulares.

## 2.4 Interpretación de los resultados

Con las tablas limpias, el filtrado, el ordenamiento y la agrupación permiten describir los hábitos de compra con cifras confiables:

1. **Cuándo se compra.** La demanda se concentra en la mañana y la tarde: el pico es a las 10 h (40.578 pedidos) y la celda más activa de la semana es el lunes a las 10 h. El domingo es el día con más pedidos (84.090) y su pico se desplaza a las 13–15 h.
2. **Con qué frecuencia se vuelve.** La espera más común después de 30 días es exactamente 7 días (9,31 % de los pedidos): hay un hábito semanal. Entre semana la mediana es de 7 días y en fin de semana de 8.
3. **Quién sostiene el negocio.** Los clientes frecuentes (19,4 %) hacen casi la mitad de los pedidos, vuelven cada 6 días y compran más por la mañana que el resto, lo que orienta las campañas de fidelización a esa franja.
4. **Cómo se llena el carrito.** Los primeros productos del carrito son los que más se recompran (67,9 % en la primera posición), así que el carrito empieza por la lista habitual del cliente.

Que todas estas cifras coincidan con las esperadas en el material confirma que la limpieza de la Lección 2 no alteró los datos: por ejemplo, los nulos de espera conservados hicieron que medias y medianas ignoren los primeros pedidos en lugar de contarlos como 0 días. Los perfiles se guardaron en `data/processed/customer_segments.csv` y `data/processed/segment_time_profile.csv`.